In [6]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.5.1+cu121
True
NVIDIA GeForce RTX 3070 Ti Laptop GPU


In [7]:
from pathlib import Path

DATA_ROOT = Path(r"E:\桌面\各种作业的东西\人工智能微专业\计算机视觉基础\交大视觉印象数据集2026\image_retrieval")

base_dir = DATA_ROOT / "base"
query_dir = DATA_ROOT / "query"

print(base_dir.exists(), query_dir.exists())

query_imgs = list(query_dir.glob("*.*"))
base_imgs = list(base_dir.rglob("*.*"))

print("query数量:", len(query_imgs))
print("base数量:", len(base_imgs))
print("前5个query:", [p.name for p in query_imgs[:5]])
print("前5个base:", [p.name for p in base_imgs[:5]])

False False
query数量: 0
base数量: 0
前5个query: []
前5个base: []


In [5]:
!pip install tqdm

In [8]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
import numpy as np

In [9]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

base_imgs = sorted([p for p in base_dir.rglob("*") if p.suffix.lower() in IMG_EXTS])
query_imgs = sorted([p for p in query_dir.rglob("*") if p.suffix.lower() in IMG_EXTS])

def get_label(path):
    return path.stem.split("-")[0]

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

class ImagePathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform
        
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), str(path)

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

weights = models.ResNet50_Weights.IMAGENET1K_V2
model = models.resnet50(weights=weights)
model.fc = nn.Identity()
model = model.to(device)
model.eval()

device

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\施琳/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth
100.0%


'cuda'

In [12]:
@torch.no_grad()
def extract_features(paths, batch_size=64):
    ds = ImagePathDataset(paths, transform)
    dl = DataLoader(ds,
                    batch_size=batch_size,
                    shuffle=False,
                    num_workers=0)

    feats = []
    out_paths = []

    for imgs, batch_paths in dl:
        imgs = imgs.to(device)

        f = model(imgs)

        # L2归一化
        f = torch.nn.functional.normalize(f, dim=1)

        feats.append(f.cpu().numpy())
        out_paths.extend(batch_paths)

    feats = np.concatenate(feats, axis=0)

    return feats, [Path(p) for p in out_paths]

In [13]:
base_feats, base_paths = extract_features(base_imgs, batch_size=64)
query_feats, query_paths = extract_features(query_imgs, batch_size=64)

print(base_feats.shape)
print(query_feats.shape)

UnidentifiedImageError: cannot identify image file 'E:\\桌面\\各种作业的东西\\人工智能微专业\\计算机视觉基础\\交大视觉印象数据集2026\\image_retrieval\\base\\BJTU\\ty-1746580766456.jpg'

In [14]:
from PIL import Image, UnidentifiedImageError

def filter_valid_images(paths):
    valid = []
    bad = []
    for p in paths:
        try:
            with Image.open(p) as img:
                img.verify()
            valid.append(p)
        except Exception:
            bad.append(p)
    return valid, bad

base_imgs, bad_base_imgs = filter_valid_images(base_imgs)
query_imgs, bad_query_imgs = filter_valid_images(query_imgs)

print("有效base:", len(base_imgs))
print("坏base:", len(bad_base_imgs))
print("有效query:", len(query_imgs))
print("坏query:", len(bad_query_imgs))
print("坏图:", [p.name for p in bad_base_imgs[:10]])

有效base: 7727
坏base: 1
有效query: 135
坏query: 0
坏图: ['ty-1746580766456.jpg']


In [15]:
base_feats, base_paths = extract_features(base_imgs, batch_size=64)
query_feats, query_paths = extract_features(query_imgs, batch_size=64)

print(base_feats.shape)
print(query_feats.shape)

(7727, 2048)
(135, 2048)


In [16]:
similarity = query_feats @ base_feats.T
print(similarity.shape)

(135, 7727)


In [17]:
idx = 0
top5 = similarity[idx].argsort()[-5:][::-1]

print("query:", query_paths[idx].name)
for rank, i in enumerate(top5, 1):
    print(rank, base_paths[i].name)

query: fhy-12k4jb1k421.jpg
1 fhy-cbuehvagvygv15.jpg
2 yk-fe700194fc.jpg
3 fhy-ikl12n12lo.jpg
4 tsg-rcnevr93rv72978v7buwueut.jpg
5 fhy-nghopnk56s.jpg


In [18]:
LANDMARKS = ["fhy","jx", "kx","mh","nm","sjz","sy","tsg","ty","yf","yk","zx"]

def get_label(path):
    return path.stem.split("-")[0]

query_labels = np.array([get_label(p) for p in query_paths])
base_labels = np.array([get_label(p) for p in base_paths])

Ks = [1, 5, 10, 20, 40, 60]

results = {lm: [] for lm in LANDMARKS}

for lm in LANDMARKS:
    q_indices = np.where(query_labels == lm)[0]
    for K in Ks:
        ps = []
        for qi in q_indices:
            topk = similarity[qi].argsort()[-K:][::-1]
            correct = np.sum(base_labels[topk] == query_labels[qi])
            ps.append(correct / K)
        results[lm].append(np.mean(ps) if len(ps) > 0 else np.nan)

results

{'fhy': [np.float64(1.0),
  np.float64(0.9600000000000001),
  np.float64(0.9133333333333334),
  np.float64(0.9033333333333334),
  np.float64(0.8716666666666666),
  np.float64(0.8366666666666666)],
 'jx': [np.float64(0.8571428571428571),
  np.float64(0.8285714285714285),
  np.float64(0.7285714285714285),
  np.float64(0.6428571428571429),
  np.float64(0.5678571428571428),
  np.float64(0.5119047619047619)],
 'kx': [np.float64(1.0),
  np.float64(0.30000000000000004),
  np.float64(0.2),
  np.float64(0.125),
  np.float64(0.07500000000000001),
  np.float64(0.058333333333333334)],
 'mh': [np.float64(1.0),
  np.float64(0.9571428571428572),
  np.float64(0.9142857142857144),
  np.float64(0.8678571428571428),
  np.float64(0.8053571428571428),
  np.float64(0.7571428571428571)],
 'nm': [np.float64(0.9333333333333333),
  np.float64(0.8666666666666667),
  np.float64(0.8666666666666667),
  np.float64(0.8433333333333334),
  np.float64(0.8233333333333331),
  np.float64(0.7933333333333332)],
 'sjz': [np.f

In [20]:
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

print(output_dir.resolve())

C:\Users\施琳\outputs


In [21]:
import matplotlib.pyplot as plt

Ks = [1, 5, 10, 20, 40, 60]

for lm in LANDMARKS:
    plt.figure(figsize=(4,3))
    plt.plot(Ks, results[lm], marker='o')
    plt.title(f"P@K - {lm}")
    plt.xlabel("K")
    plt.ylabel("Precision")
    plt.ylim(0, 1.05)
    plt.grid(True)

    plt.savefig(output_dir / f"P@K_{lm}.png")

    plt.close()

print("完成，共保存", len(LANDMARKS), "张图")

完成，共保存 12 张图


In [3]:
%pip install paddleocr paddlepaddle

  Using cached paddleocr-3.7.0-py3-none-any.whl.metadata (28 kB)
  Using cached paddlex-3.7.1-py3-none-any.whl.metadata (80 kB)
  Using cached aistudio_sdk-0.3.8-py3-none-any.whl.metadata (1.1 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
  Using cached huggingface_hub-1.19.0-py3-none-any.whl.metadata (14 kB)
  Using cached modelscope-1.37.1-py3-none-any.whl.metadata (43 kB)
  Using cached prettytable-3.17.0-py3-none-any.whl.metadata (34 kB)
  Using cached py_cpuinfo-9.0.0-py3-none-any.whl.metadata (794 bytes)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached imagesize-2.0.0-py2.py3-none-any.whl.metadata (1.5 kB)
  Using cached opencv_contrib_python-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached pypdfium2-5.9.0-py3-none-win_amd64.whl.metadata (68 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached opt_einsum-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached safetensors-0.8.0-cp3

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [pandas]
   --------------------------- ------------ 42/61 [panda

In [5]:
%pip install -U typing_extensions

Note: you may need to restart the kernel to use updated packages.


In [1]:
from paddleocr import PaddleOCR

D:\miniconda\envs\torch-gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from PIL import Image, ImageDraw
from pathlib import Path
import matplotlib.pyplot as plt

ocr = PaddleOCR(use_angle_cls=False, lang="ch")

C:\Users\施琳\AppData\Local\Temp\ipykernel_24924\1845412054.py:5: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=False, lang="ch")
D:\miniconda\envs\torch-gpu\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `C:\Users\施琳\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Fetching 6 files: 100%|██████████| 6/

RuntimeError: [json.exception.parse_error.101] parse error at line 1, column 1: attempting to parse an empty input; check that your input string or stream contains the expected JSON

In [2]:
%pip install easyocr

  Using cached easyocr-1.7.2-py3-none-any.whl.metadata (10 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached scikit_image-0.25.2-cp310-cp310-win_amd64.whl.metadata (14 kB)
  Using cached ninja-1.13.0-py3-none-win_amd64.whl.metadata (5.1 kB)
  Using cached imageio-2.37.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached lazy_loader-0.5-py3-none-any.whl.metadata (5.9 kB)
Using cached easyocr-1.7.2-py3-none-any.whl (2.9 MB)
Using cached ninja-1.13.0-py3-none-win_amd64.whl (309 kB)
Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl (40.1 MB)
Using cached scikit_image-0.25.2-cp310-cp310-win_amd64.whl (12.8 MB)
Using cached imageio-2.37.3-py3-none-any.whl (317 kB)
Using cached lazy_loader-0.5-py3-none-any.whl (8.0 kB)

   ---------------------------------------- 0/6 [opencv-python-headless]
   ---------------------------------------- 0/6 [opencv-python-headless]
   ---------------------------------------- 0/6 [openc

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import easyocr
from PIL import Image, ImageDraw
from pathlib import Path

reader = easyocr.Reader(['ch_sim', 'en'], gpu=True)

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [9]:
from pathlib import Path

DATA_ROOT = Path(r"E:\桌面\各种作业的东西\人工智能微专业\计算机视觉基础\实验 阶段2\交大视觉印象数据集2026")

base_dir = DATA_ROOT / "image_retrieval" / "base"
query_dir = DATA_ROOT / "image_retrieval" / "query"
det_data_dir = DATA_ROOT / "object_detection" / "data"

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

base_paths = sorted([p for p in base_dir.rglob("*") if p.suffix.lower() in IMG_EXTS])
query_paths = sorted([p for p in query_dir.rglob("*") if p.suffix.lower() in IMG_EXTS])
det_img_paths = sorted([p for p in det_data_dir.rglob("*") if p.suffix.lower() in IMG_EXTS])

print("base:", len(base_paths))
print("query:", len(query_paths))
print("det_data:", len(det_img_paths))
print(base_paths[0])

base: 7728
query: 135
det_data: 1494
E:\桌面\各种作业的东西\人工智能微专业\计算机视觉基础\实验 阶段2\交大视觉印象数据集2026\image_retrieval\base\BJTU\fhy-0120240508151840.jpg


In [16]:
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path

def draw_easyocr_boxes(img_path, save_path, conf_thres=0.3):
    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)

    result = reader.readtext(img_np)

    draw = ImageDraw.Draw(img)
    kept = []

    for box, text, conf in result:
        if conf < conf_thres:
            continue

        pts = [(int(x), int(y)) for x, y in box]
        draw.line(pts + [pts[0]], width=4)
        kept.append((box, text, conf))

    img.save(save_path)
    return kept, result

In [17]:
preview_dir = Path("outputs/text_detection_preview")
preview_dir.mkdir(parents=True, exist_ok=True)

LANDMARKS = ["fhy","jx", "kx","mh","nm","sjz","sy","tsg","ty","yf","yk","zx"]

def get_label(path):
    return path.stem.split("-")[0]

good_candidates = []

for lm in LANDMARKS:
    imgs = [p for p in det_img_paths if get_label(p) == lm]
    print("\n类别:", lm, "图片数:", len(imgs))

    count = 0
    for img_path in imgs[:30]:  # 每类先试前30张
        save_path = preview_dir / f"{lm}_{img_path.stem}_easyocr.jpg"
        kept, raw = draw_easyocr_boxes(img_path, save_path, conf_thres=0.3)

        if len(kept) > 0:
            print("  有检测:", img_path.name, "有效框数:", len(kept))
            good_candidates.append(img_path)
            count += 1

        if count >= 3:
            break

print("\n候选图数量:", len(good_candidates))


类别: fhy 图片数: 249
  有检测: fhy-04qgjnsjosp902psvnz.png 有效框数: 1
  有检测: fhy-0589095011.png 有效框数: 1
  有检测: fhy-09djfb.png 有效框数: 1

类别: jx 图片数: 18
  有检测: jx-12bn3jol13.jpg 有效框数: 2
  有检测: jx-1nl3llnk.jpg 有效框数: 1
  有检测: jx-788bdc05cbd02a07b9ffd3018211af82.jpg 有效框数: 1

类别: kx 图片数: 47
  有检测: kx-4o59hbwuovsb.jpg 有效框数: 1
  有检测: kx-asd3124ds.jpg 有效框数: 2
  有检测: kx-gfkjdfwieuf2o.jpg 有效框数: 1

类别: mh 图片数: 245
  有检测: mh-00iii0wer.jpg 有效框数: 1
  有检测: mh-091fbnqpasdm.jpeg 有效框数: 2
  有检测: mh-1746580766618.jpg 有效框数: 2

类别: nm 图片数: 143
  有检测: nm-103br8f0peh9g.jpeg 有效框数: 1
  有检测: nm-10m260jh860.jpg 有效框数: 2
  有检测: nm-1c2d3e4f5g6h.jpg 有效框数: 1

类别: sjz 图片数: 142
  有检测: sjz-1e2f3g4h5i6j.jpeg 有效框数: 1
  有检测: sjz-202405152208431.jpg 有效框数: 2
  有检测: sjz-202405152208441.jpg 有效框数: 2

类别: sy 图片数: 234
  有检测: sy-1446ajhd43.jpg 有效框数: 2
  有检测: sy-1a2b3c4d5e6f.jpeg 有效框数: 1
  有检测: sy-202405152208451.jpg 有效框数: 2

类别: tsg 图片数: 80
  有检测: tsg-01.png 有效框数: 1
  有检测: tsg-04.jpg 有效框数: 1
  有检测: tsg-20240515220851.jpg 有效框数: 4

类别: ty 图片数: 

In [18]:
selected = []

for lm in LANDMARKS:
    lm_good = [p for p in good_candidates if get_label(p) == lm]
    chosen = lm_good[:2]
    selected.extend(chosen)
    print(lm, "最终选择:", [p.name for p in chosen])

print("总计:", len(selected))

fhy 最终选择: ['fhy-04qgjnsjosp902psvnz.png', 'fhy-0589095011.png']
jx 最终选择: ['jx-12bn3jol13.jpg', 'jx-1nl3llnk.jpg']
kx 最终选择: ['kx-4o59hbwuovsb.jpg', 'kx-asd3124ds.jpg']
mh 最终选择: ['mh-00iii0wer.jpg', 'mh-091fbnqpasdm.jpeg']
nm 最终选择: ['nm-103br8f0peh9g.jpeg', 'nm-10m260jh860.jpg']
sjz 最终选择: ['sjz-1e2f3g4h5i6j.jpeg', 'sjz-202405152208431.jpg']
sy 最终选择: ['sy-1446ajhd43.jpg', 'sy-1a2b3c4d5e6f.jpeg']
tsg 最终选择: ['tsg-01.png', 'tsg-04.jpg']
ty 最终选择: ['ty-02djdb.png', 'ty-20240515220845.jpg']
yf 最终选择: ['yf-cxvbhjvsbfdiw.jpg', 'yf-dnsjfgki43534.jpg']
yk 最终选择: ['yk-20260511181541_151_105.jpg', 'yk-2gvfjc.jpg']
zx 最终选择: ['zx-09ucbc.png', 'zx-123n1l3kn1l3.png']
总计: 24


In [19]:
final_dir = Path("outputs/text_detection_final")
final_dir.mkdir(parents=True, exist_ok=True)

summary = []

for img_path in selected:
    lm = get_label(img_path)
    save_path = final_dir / f"{lm}_{img_path.stem}_easyocr.jpg"

    kept, raw = draw_easyocr_boxes(img_path, save_path, conf_thres=0.3)

    summary.append({
        "landmark": lm,
        "image": img_path.name,
        "valid_boxes": len(kept),
        "raw_boxes": len(raw),
        "output": str(save_path)
    })

print("完成，保存到:", final_dir.resolve())
print("共生成:", len(summary), "张")

完成，保存到: C:\Users\施琳\outputs\text_detection_final
共生成: 24 张


In [20]:
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

test_img = det_img_paths[0]
img = Image.open(test_img).convert("RGB")
img_np = np.array(img)

horizontal_list, free_list = reader.detect(img_np)

print("horizontal:", horizontal_list)
print("free:", free_list)

horizontal: [[[np.int32(677), np.int32(693), np.int32(485), np.int32(513)], [np.int32(134), np.int32(162), np.int32(594), np.int32(698)], [np.int32(170), np.int32(279), np.int32(389), np.int32(767)]]]
free: [[]]


In [21]:
def draw_easyocr_detect_only(img_path, save_path):
    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)

    horizontal_list, free_list = reader.detect(img_np)

    draw = ImageDraw.Draw(img)
    count = 0

    # horizontal_list: [ [x_min, x_max, y_min, y_max], ... ]
    if horizontal_list and horizontal_list[0]:
        for box in horizontal_list[0]:
            x_min, x_max, y_min, y_max = map(int, box)
            draw.rectangle([x_min, y_min, x_max, y_max], width=4)
            count += 1

    # free_list: 四点框
    if free_list and free_list[0]:
        for box in free_list[0]:
            pts = [(int(x), int(y)) for x, y in box]
            draw.line(pts + [pts[0]], width=4)
            count += 1

    img.save(save_path)
    return count

In [22]:
preview_dir = Path("outputs/text_detection_detectonly_preview")
preview_dir.mkdir(parents=True, exist_ok=True)

good_candidates = []

for lm in LANDMARKS:
    imgs = [p for p in det_img_paths if get_label(p) == lm]
    print("\n类别:", lm, "图片数:", len(imgs))

    count_good = 0
    for img_path in imgs[:80]:  
        save_path = preview_dir / f"{lm}_{img_path.stem}_detectonly.jpg"
        box_count = draw_easyocr_detect_only(img_path, save_path)

        if box_count > 0:
            print("  有检测:", img_path.name, "框数:", box_count)
            good_candidates.append(img_path)
            count_good += 1

        if count_good >= 5:
            break

print("候选图数量:", len(good_candidates))


类别: fhy 图片数: 249
  有检测: fhy-0120240508151840.jpg 框数: 3
  有检测: fhy-04qgjnsjosp902psvnz.png 框数: 2
  有检测: fhy-0589095011.png 框数: 2
  有检测: fhy-09djfb.png 框数: 5
  有检测: fhy-1746580766808.jpg 框数: 2

类别: jx 图片数: 18
  有检测: jx-12bn3jol13.jpg 框数: 5
  有检测: jx-1nl3llnk.jpg 框数: 3
  有检测: jx-788bdc05cbd02a07b9ffd3018211af82.jpg 框数: 1
  有检测: jx-c415cdae3732c6339cbf8a0a01ffd084.jpg 框数: 1
  有检测: jx-dbd05bf5-cec6-4c8a-858b-49feabafe2fe.png 框数: 4

类别: kx 图片数: 47
  有检测: kx-1ac1815f-6535-4968-93f7-45ba326e7a56.png 框数: 1
  有检测: kx-1vgyuhy.jpg 框数: 1
  有检测: kx-20260511140016_128_105(1).jpg 框数: 1
  有检测: kx-20260511140021_133_105(1).jpg 框数: 4
  有检测: kx-34133413e1c563fe28c70f812d5d6408.jpg 框数: 3

类别: mh 图片数: 245
  有检测: mh-00iii0wer.jpg 框数: 2
  有检测: mh-0513a4fad192097469b24a235baf8312.jpg 框数: 3
  有检测: mh-091fbnqpasdm.jpeg 框数: 3
  有检测: mh-092r0924r2owof.jpg 框数: 2
  有检测: mh-0b18e68c28c84bb1c8d2c75e43081deb.jpg 框数: 2

类别: nm 图片数: 143
  有检测: nm-103br8f0peh9g.jpeg 框数: 2
  有检测: nm-10m260jh860.jpg 框数: 3
  有检测: nm-1437wdx

In [23]:
selected = []

for lm in LANDMARKS:
    lm_good = [p for p in good_candidates if get_label(p) == lm]
    chosen = lm_good[:2]
    selected.extend(chosen)
    print(lm, "最终选择:", [p.name for p in chosen])

print("总计:", len(selected))

fhy 最终选择: ['fhy-0120240508151840.jpg', 'fhy-04qgjnsjosp902psvnz.png']
jx 最终选择: ['jx-12bn3jol13.jpg', 'jx-1nl3llnk.jpg']
kx 最终选择: ['kx-1ac1815f-6535-4968-93f7-45ba326e7a56.png', 'kx-1vgyuhy.jpg']
mh 最终选择: ['mh-00iii0wer.jpg', 'mh-0513a4fad192097469b24a235baf8312.jpg']
nm 最终选择: ['nm-103br8f0peh9g.jpeg', 'nm-10m260jh860.jpg']
sjz 最终选择: ['sjz-10448bd820d0602c50b232c97b6f5a43.jpg', 'sjz-16165huihuh.jpg']
sy 最终选择: ['sy-015kcv.png', 'sy-09f9f9f9f.jpg']
tsg 最终选择: ['tsg-01.png', 'tsg-04.jpg']
ty 最终选择: ['ty-02djdb.png', 'ty-1455ihdd781.jpg']
yf 最终选择: ['yf-cxvbhjvsbfdiw.jpg', 'yf-dnsjfgki43534.jpg']
yk 最终选择: ['yk-1bvjiji.jpg', 'yk-20260511181541_151_105.jpg']
zx 最终选择: ['zx-0924r9420fwf.jpg', 'zx-09ucbc.png']
总计: 24


In [24]:
final_dir = Path("outputs/text_detection_detectonly_final")
final_dir.mkdir(parents=True, exist_ok=True)

summary = []

for img_path in selected:
    lm = get_label(img_path)
    save_path = final_dir / f"{lm}_{img_path.stem}_detectonly.jpg"

    box_count = draw_easyocr_detect_only(img_path, save_path)

    summary.append({
        "landmark": lm,
        "image": img_path.name,
        "box_count": box_count,
        "output": str(save_path)
    })

print("完成，保存到:", final_dir.resolve())
print("共生成:", len(summary), "张")

完成，保存到: C:\Users\施琳\outputs\text_detection_detectonly_final
共生成: 24 张
